# FoX: latest-answer loss versus all-token training

**Two losses, identical examples and initial model functions.** Train on short latest-update histories, then vary target lag and older-prefix length independently. A separate control grid tests which restrictions matter.

This notebook defaults to **real Colab training** (`PROFILE="colab"`). `smoke` is a two-update software check only. The core uses a small controlled vocabulary; the optional real-text wrapper is explicitly semi-synthetic. Read [the design](../docs/loss_and_constraints_study.md) before interpreting a radius.

The latest-answer objective adapts the paper's answer supervision to the same full-vocabulary decoder used for all-token cross-entropy. It does not reproduce every theorem assumption or assume that SGD's cutoff must equal four.

## 1. Open your repository and select a GPU
Set `REPO_URL` for Colab. Implementation lives in Python modules, not hidden notebook cells.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # After publishing: https://github.com/YOUR_NAME/fox_experiments.git
REPO_REF = "main"  # Branch, tag, or commit to use in a new Colab clone.
candidates = [Path.cwd(), Path.cwd().parent, Path("/content/fox_experiments")]
REPO = next((p for p in candidates if (p / "src/fox_experiments").is_dir()), None)
if REPO is None:
    if not REPO_URL:
        raise ValueError("Set REPO_URL above, or open this notebook inside a local clone.")
    REPO = Path("/content/fox_experiments")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", REPO_REF])
REPO = REPO.resolve()
os.chdir(REPO)
print("Repository:", REPO)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
      if (REPO / ".git/HEAD").exists() and subprocess.run(
          ["git", "rev-parse", "--verify", "HEAD"], capture_output=True).returncode == 0
      else "No commit yet: commit the source before a scientific run.")

In [ ]:
if os.environ.get("FOX_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
sys.path.insert(0, str(REPO / "src"))  # Make this checkout visible to the current kernel.

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print("PyTorch:", torch.__version__, "| Device:", DEVICE)

## 2. Choose the experiment

Run `baseline` first. Change **one** `CONDITION` and use a new run name for a diagnostic comparison. The factorized/direct × loss × optimizer grid is already included. All-token weights are one in baseline.

The primary run uses 12 branches × 2,000 updates × 32 histories = **768,000 training histories** (before optional tuning). Variable sequence lengths mean token counts differ from history counts. The notebook prints actual settings; it does not promise a GPU runtime.

The `text_background` mode uses natural text around controlled update sentences and requires ordinary causal routing. Compare it with a `records` run using the same causal routing first, to avoid attributing a parser change to data alone.

In [ ]:
from dataclasses import replace
from fox_experiments.loss_study.config import LossStudyConfig, branches, intervention
from IPython.display import display, Image
import pandas as pd

PROFILE = os.environ.get("FOX_LOSS_PROFILE", "colab")
CONDITION = os.environ.get("FOX_LOSS_CONDITION", "baseline")
DATA_MODE = "records"  # records / text_background
RUN_NAME = os.environ.get("FOX_LOSS_RUN_NAME", PROFILE + "_" + CONDITION + "_v1")
USE_GOOGLE_DRIVE = False
RESUME = False
RUN_TRAINING = True
RUN_LR_TUNING = False  # A separate tuned comparison; baseline keeps rates matched.
WORKSPACE = Path(os.environ.get("FOX_LOSS_WORKSPACE", str(REPO / "outputs/loss_study")))
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/fox_experiments/loss_study")
OUT = WORKSPACE / RUN_NAME
DATA_DIR = Path(os.environ.get("FOX_DATA_DIR", str(REPO / "data/longcrawl64_pilot")))
CONFIG = intervention(LossStudyConfig.for_profile(PROFILE), CONDITION)
if DATA_MODE == "text_background":
    CONFIG = replace(CONFIG, data_mode=DATA_MODE, routing="causal", batch_size=4)
display(pd.DataFrame(branches(CONFIG)))
print("Training histories:", len(branches(CONFIG)) * CONFIG.steps * CONFIG.batch_size)
display(pd.Series(CONFIG.to_dict(), name="Setting").to_frame())
if RUN_TRAINING and PROFILE != "smoke" and DEVICE != "cuda":
    raise RuntimeError("Select a Colab GPU, or explicitly choose PROFILE='smoke'.")

## 3. Inspect the data and the loss positions

`answer_only` scores the final latest-value answer. `all_tokens` scores every non-padding next token. Both models see the same prefixes. The generator does not feed gold record positions to the model.

Short qualification includes latest-value, stale-value, unrelated-value, and alternative-query edits where applicable. A changed query has its own actual lag; metadata distinguishes this from the base history's panel label.

In [ ]:
from fox_experiments.loss_study.data import StudyData

if CONFIG.data_mode == "text_background":
    from fox_experiments.cli import prepare_data
    _, manifest = prepare_data(DATA_DIR)
    print("Real-text corpus SHA256:", manifest["sha256"])
DATA = StudyData(CONFIG, data_dir=DATA_DIR if CONFIG.data_mode == "text_background" else None)
display(DATA.describe())
preview = DATA.batch(step=0, seed=CONFIG.seeds[0])
for example in preview["examples"][:3]:
    print(DATA.render_example(example))
print("Answer targets:", int(preview["answer_mask"].sum()))
print("All non-padding targets:", int(preview["targets"].ne(-100).sum()))

## 4. Optional short-only learning-rate search

Leave this off for the clean **same-rate objective intervention**. If an optimizer fails short acquisition, run a separately named tuned study. Every branch receives the same candidate count and trial budget; selection uses only a separate short-validation namespace, never the long test. A selected rate is not a guarantee of acquisition. Resume a tuned run from its saved `selected_config.json` instead of retuning.

In [ ]:
from fox_experiments.loss_study.tuning import tune_learning_rates

if RUN_LR_TUNING:
    if RESUME:
        CONFIG = LossStudyConfig.from_json(WORKSPACE / (RUN_NAME + "_tuning") / "selected_config.json")
    else:
        CONFIG = tune_learning_rates(
            CONFIG, WORKSPACE / (RUN_NAME + "_tuning"), device=DEVICE,
            data_dir=DATA_DIR if CONFIG.data_mode == "text_background" else None,
            steps=2 if PROFILE == "smoke" else 250,
            multipliers=(1 / 3, 1.0, 3.0),
        )
    print("This is the independently tuned comparison.")
    display(pd.read_csv(WORKSPACE / (RUN_NAME + "_tuning") / "trials.csv"))

## 5. Run the two experiments

Initial models and all later checkpoints are evaluated. Failed short qualification is recorded explicitly; the branch continues so we can see whether it acquires the rule later. Its long scores remain descriptive until it qualifies. Optimizer moments are saved and retained on resume.

Gradient diagnostics compute answer and non-answer contributions separately for inspection; the optimizer still takes **one update on the chosen total loss**. Epsilon/moment ratios show whether the Adam intervention is numerically active.

In [ ]:
from fox_experiments.loss_study import run_loss_study

if RUN_TRAINING:
    artifacts = run_loss_study(
        CONFIG, OUT, device=DEVICE,
        data_dir=DATA_DIR if CONFIG.data_mode == "text_background" else None,
        resume=RESUME,
    )
    print(artifacts)
else:
    print("Reading the existing run:", OUT)

## 6. Diagnose before claiming generalization

1. **Short joint-edit accuracy:** did the branch learn the rule, including key dependence?
2. **Answer probability:** correct argmax and confident retrieval are different criteria.
3. **Lag × older prefix:** distinguish distant-target failure from stale-memory failure.
4. **Range over checkpoints:** is the tested range expanding, or merely stable?
5. **Gradient conflict:** negative answer/non-answer cosine suggests competing updates; a small weighted answer gradient indicates dilution.
6. **Gate and moment diagnostics:** inspect local binding, retrieval decay, and epsilon activity.

A missing range means unqualified short learning. Passing the largest tested lag means **at least that grid ceiling**. Neither result certifies arbitrary length. Results on the sparse lag grid do not certify skipped lags.

In [ ]:
from fox_experiments.loss_study.analysis import summarize_loss_study
import json

report = summarize_loss_study(OUT)
print(report)
for name in ("status.json", "failures.json"):
    if (OUT / name).exists():
        print(name, json.loads((OUT / name).read_text()))
for name in ("branches.csv", "summary.csv", "training.csv", "gradients.csv"):
    path = OUT / name
    if path.exists():
        frame = pd.read_csv(path)
        if name == "summary.csv":
            display(frame.sort_values("step").groupby("branch").tail(1))
        else:
            display(frame if name == "branches.csv" else frame.tail(24))
for figure in sorted(OUT.rglob("*.png")):
    display(Image(filename=str(figure)))

### Compare saved constraint runs

After completing an additional condition, add its output directory below. The comparison lists every changed setting, pairs seeds/gates/optimizers/objectives, and leaves radius differences missing for unqualified, incomplete, or censored branches. A positive finite-grid difference is descriptive evidence at this budget; it does not prove necessity.

In [ ]:
from fox_experiments.loss_study.comparison import compare_conditions

SAVED_CONDITIONS = {}  # Example: {"baseline": OUT, "causal": WORKSPACE / "colab_unrestricted_routing_v1"}
if SAVED_CONDITIONS:
    comparison = compare_conditions(
        SAVED_CONDITIONS, baseline="baseline",
        destination=WORKSPACE / "condition_comparison.csv",
    )
    display(comparison)

## 7. Optional closer scalar reference

This separate reference uses the existing two-stage binding model and answer loss, with explicit scalar diagnostics. Its practical acquisition/schedules differ from the theorem, as recorded in its configuration. It is **not** the all-token neural experiment and should not be merged into its plots.

In [ ]:
RUN_SCALAR_REFERENCE = False
if RUN_SCALAR_REFERENCE:
    from fox_experiments.mechanism import run_mechanism
    scalar_artifacts = run_mechanism(
        WORKSPACE / (RUN_NAME + "_scalar_reference"),
        profile="smoke" if PROFILE == "smoke" else "pilot",
        device=DEVICE, seeds=CONFIG.seeds, recall_lags=(2,),
    )
    print(scalar_artifacts)

## 8. Export reports

The ZIP includes configuration, provenance, raw measurements, failure records, and plots. Large weight files stay in the run directory; use Drive to retain them across Colab disconnects. Set `include_checkpoints=True` only if you want a larger archive.

Suggested progression: `baseline` → `unrestricted_routing` → one diagnosis-driven condition → fresh-seed confirmation. Run the real-text extension after short retrieval works. Do not discard failed seeds or select restrictions using the final long test.

In [ ]:
from fox_experiments.notebook_utils import archive_results

folders = {"loss_study": OUT}
tuning_folder = WORKSPACE / (RUN_NAME + "_tuning")
if tuning_folder.exists():
    folders["short_only_tuning"] = tuning_folder
archive = archive_results(WORKSPACE / (RUN_NAME + "_results.zip"), folders)
print("Saved:", archive)